In [ ]:
%pip install --upgrade openai pydantic mermaid-py

# Docs Watcher Agent: Long-Running Loops with State

In this notebook you will build a stateful agent that monitors documentation pages for changes, classifies their severity using both heuristic rules and LLM-based analysis, and maintains persistent memory across ticks. This pattern teaches the fundamentals of long-running agent loops with state persistence.

## Architecture Overview

This notebook teaches **long-running agent patterns** for monitoring and alerting. Unlike request-response agents that answer a single question and stop, a docs watcher agent runs continuously (or on a schedule), maintaining state between executions.

We demonstrate two approaches:

- **Approach 1: Stateful Worker (Pull-Based)** Polls pages on a schedule, diffs content against the last known snapshot, and stores state in a database. This is the most common pattern for monitoring external pages you do not control.

- **Approach 2: Push-Based Mechanism** Simulates a webhook/event-driven processing model. Instead of polling, the agent receives notifications when content changes. This is more efficient but requires infrastructure support.

### Poll-Based Loop Architecture

Each "tick" of the agent follows this flow. The agent is designed so that any tick can be re-run safely without producing duplicate events.

In [ ]:
import mermaid as md

md.Mermaid("""
flowchart LR
    A[Schedule / Tick] --> B[Fetch Page]
    B --> C[Hash Content]
    C --> D{Compare to Last Hash}
    D -- No Change --> E[Log & Skip]
    D -- Change Detected --> F[Compute Diff]
    F --> G[Classify Severity]
    G --> H[Store Event]
    H --> I[Notify / Alert]
""")

## Key Concepts in Detail

### Idempotency
Running the same tick twice produces the same result; no duplicate alerts. We achieve this by tracking the last hash we alerted on per source. If the current content hash matches the last alerted hash, we skip alerting even though a change was previously detected.

### Content Hashing
We use SHA-256 to detect changes efficiently. Instead of comparing entire page contents character-by-character, we hash the content and compare short fixed-length strings. This is fast and reliable for change detection.

### Diffing
When a change IS detected, we compute a unified diff showing exactly what lines were added, removed, or modified. This gives the severity classifier (and the human operator) precise context about what changed.

### Severity Classification
We use a two-tier approach:
1. **Heuristic rules**: Fast keyword matching (e.g., "deprecated" = HIGH, "new feature" = MEDIUM)
2. **LLM classification**: For more nuanced understanding, we can optionally send the diff to GPT-5-mini for classification

### Operational Memory
For each source, we track: last content hash, failure count, last fetch time, and last alerted hash. This is stored in a key-value table in SQLite.

### Backoff
If a source fails 3 or more times consecutively, the agent skips it on future ticks. This prevents hammering a broken endpoint and wasting resources.

## Setup

Let's start by importing the libraries we need and defining the Pydantic model for structured severity classification. Notice how we use `SeverityClassification` to guarantee that the LLM always returns a well-typed response.

In [ ]:
import os
import json
import hashlib
import sqlite3
import pathlib
import difflib
from datetime import datetime
from pydantic import BaseModel, Field
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-5-mini"


class SeverityClassification(BaseModel):
    """Structured output for change severity classification."""
    severity: str = Field(description="Severity level: HIGH, MEDIUM, or LOW")
    reason: str = Field(description="Brief explanation for the severity rating")

## Load Simulated Content from Resources

Since we cannot guarantee that real documentation pages will change during the execution of this notebook, we load simulated content from a resource file. This gives us deterministic control over when "changes" occur, making the demo fully reproducible.

In production, the `fetch_content` function would make real HTTP requests to the monitored URLs. Here, we load everything from a JSON file that ships with this notebook.

In [ ]:
RESOURCES = pathlib.Path("resources")
watcher_data = json.loads((RESOURCES / "data" / "docs_watcher_content.json").read_text())

# Preview what we loaded
print(f"Loaded {len(watcher_data['sources'])} sources and "
      f"{len(watcher_data['simulated_content'])} simulated content entries.")
print()
for src in watcher_data["sources"]:
    print(f"  [{src['id']}] {src['name']} -> {src['url']}")

## Create the State Database

The agent needs persistent storage to remember what it has seen. We use SQLite with four tables:

- **sources**: The documentation pages we monitor (name, URL, active flag)
- **snapshots**: Every fetched version of a page (content hash, full text, timestamp)
- **events**: Detected changes with severity classification and diff
- **kv**: A key-value store for per-source operational metadata (last alerted hash, failure count, etc.)

We use an in-memory database for this demo, but in production you would use a file-based SQLite database or a more robust store like PostgreSQL or Redis.

In [ ]:
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row

conn.executescript("""
CREATE TABLE sources (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    url TEXT NOT NULL,
    active BOOLEAN DEFAULT 1
);

CREATE TABLE snapshots (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_id INTEGER,
    fetched_at TEXT,
    content_hash TEXT,
    content_text TEXT,
    FOREIGN KEY (source_id) REFERENCES sources(id)
);

CREATE TABLE events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_id INTEGER,
    created_at TEXT,
    severity TEXT,
    summary TEXT,
    diff_text TEXT,
    FOREIGN KEY (source_id) REFERENCES sources(id)
);

CREATE TABLE kv (
    source_id INTEGER,
    key TEXT,
    value TEXT,
    PRIMARY KEY (source_id, key),
    FOREIGN KEY (source_id) REFERENCES sources(id)
);
""")

print("Database schema created successfully.")
print("Tables: sources, snapshots, events, kv")

## Register Sources

Now let's register the documentation pages we want to monitor. We pull these directly from the resource file so that the notebook stays in sync with the data.

In [ ]:
# Register documentation sources to monitor from the loaded JSON
sources_data = [(s["id"], s["name"], s["url"]) for s in watcher_data["sources"]]
conn.executemany("INSERT INTO sources (id, name, url) VALUES (?, ?, ?)", sources_data)
conn.commit()

print("Registered sources:")
for row in conn.execute("SELECT * FROM sources").fetchall():
    print(f"  [{row['id']}] {row['name']} -> {row['url']}")

## Simulated Content Setup

With the resource data loaded, we now set up version tracking and a `fetch_content` function that returns the appropriate version of each source's content. All sources start at version `v1`. Later, you will flip specific sources to `v2` to simulate real-world page updates.

In [ ]:
# Build simulated_content dict with integer keys from the loaded JSON
simulated_content = {
    int(k): v for k, v in watcher_data["simulated_content"].items()
}

# Track which version each source is on
current_versions = {sid: "v1" for sid in simulated_content}


def fetch_content(source_id):
    """Simulate fetching page content.

    In production, this would use requests.get(url) to fetch real pages.
    For this demo, we return simulated content based on the current version.
    """
    version = current_versions.get(source_id, "v1")
    return simulated_content.get(source_id, {}).get(version, "")


print("Simulated content loaded. All sources start at version v1.")
print(f"Source 1 (OpenAI) v1 preview: {simulated_content[1]['v1'][:60]}...")
print(f"Source 2 (Anthropic) v1 preview: {simulated_content[2]['v1'][:60]}...")

## Helper Functions

These utility functions handle the core operations: hashing content, retrieving the last snapshot, computing diffs, and managing the key-value store for per-source metadata. Take a moment to read through each one; they are the building blocks you will use in the agent tick function.

In [ ]:
def compute_hash(content):
    """Compute SHA-256 hash of content for efficient change detection."""
    return hashlib.sha256(content.encode()).hexdigest()


def get_last_snapshot(source_id):
    """Retrieve the most recent snapshot for a source."""
    row = conn.execute(
        "SELECT * FROM snapshots WHERE source_id = ? ORDER BY fetched_at DESC LIMIT 1",
        (source_id,)
    ).fetchone()
    return dict(row) if row else None


def compute_diff(old_text, new_text):
    """Compute a unified diff between two text versions."""
    diff = difflib.unified_diff(
        old_text.splitlines(), new_text.splitlines(),
        lineterm='', fromfile='previous', tofile='current'
    )
    return '\n'.join(diff)


def get_kv(source_id, key, default=None):
    """Get a value from the per-source key-value store."""
    row = conn.execute(
        "SELECT value FROM kv WHERE source_id = ? AND key = ?",
        (source_id, key)
    ).fetchone()
    return json.loads(row[0]) if row else default


def set_kv(source_id, key, value):
    """Set a value in the per-source key-value store."""
    conn.execute(
        "INSERT OR REPLACE INTO kv (source_id, key, value) VALUES (?, ?, ?)",
        (source_id, key, json.dumps(value))
    )
    conn.commit()


# Quick test of helpers
test_hash = compute_hash("hello world")
print(f"SHA-256 of 'hello world': {test_hash[:24]}...")

test_diff = compute_diff("line 1\nline 2", "line 1\nline 2 modified\nline 3")
print(f"\nSample diff output:")
print(test_diff)

## Severity Classifier

We provide two classification strategies:

1. **Heuristic classifier**: Fast, no API calls, uses keyword matching. Great for initial filtering.
2. **LLM classifier**: Uses GPT-5-mini with Pydantic structured output to understand context and provide nuanced severity ratings. Better for production use where accuracy matters.

The heuristic classifier is always used as a fallback. The LLM classifier is optional and can be enabled per tick.

In [ ]:
HIGH_SEVERITY_KEYWORDS = [
    'deprecated', 'removed', 'breaking', 'migration',
    'pricing', 'rate limit', 'security'
]
MEDIUM_SEVERITY_KEYWORDS = [
    'new', 'improved', 'preview', 'beta', 'released', 'added'
]


def classify_severity_heuristic(diff_text):
    """Classify change severity using keyword matching.

    Returns HIGH if any high-severity keywords are found,
    MEDIUM if any medium-severity keywords are found,
    LOW otherwise.
    """
    diff_lower = diff_text.lower()
    for kw in HIGH_SEVERITY_KEYWORDS:
        if kw in diff_lower:
            return "HIGH"
    for kw in MEDIUM_SEVERITY_KEYWORDS:
        if kw in diff_lower:
            return "MEDIUM"
    return "LOW"


def classify_severity_llm(diff_text, source_name):
    """Classify change severity using GPT-5-mini with Pydantic structured output.

    Returns a SeverityClassification with severity and reason.
    """
    response = client.responses.create(
        model=MODEL,
        instructions="You are a documentation change classifier.",
        input=f"""Classify the severity of this documentation change for developers.

Source: {source_name}
Changes:
{diff_text}

HIGH = breaking changes, deprecations, pricing changes, security issues
MEDIUM = new features, improvements, new models
LOW = formatting, minor updates, typo fixes""",
        text={
            "format": {
                "type": "json_schema",
                "name": "severity_classification",
                "strict": True,
                "schema": SeverityClassification.model_json_schema()
            }
        }
    )
    return SeverityClassification.model_validate_json(response.output_text)


# Test the heuristic classifier
print("Heuristic classifier tests:")
print(f"  'deprecated gpt-4' -> {classify_severity_heuristic('deprecated gpt-4')}")
print(f"  'new model released' -> {classify_severity_heuristic('new model released')}")
print(f"  'fixed typo in docs' -> {classify_severity_heuristic('fixed typo in docs')}")

## Approach 1: Stateful Worker (Pull-Based)

The **stateful worker** pattern is the core of this notebook. Each call to `agent_tick()` represents one cycle of the monitoring loop:

1. Iterate through all active sources
2. Check backoff status (skip sources with too many consecutive failures)
3. Fetch current content and compute its hash
4. Compare against the last known snapshot
5. If changed: compute diff, classify severity, store event, alert
6. If unchanged: log and skip

The function is **idempotent**: running it twice with no underlying changes produces no new events. This is critical for production agents that may be triggered by unreliable schedulers.

In [ ]:
def agent_tick(use_llm_classification=False):
    """
    One tick of the docs watcher agent.
    Fetches all active sources, checks for changes, creates events.

    This function is idempotent: running it twice with no changes
    produces no new events.

    Args:
        use_llm_classification: If True, use GPT-5-mini for severity classification.
                                If False, use fast heuristic keyword matching.
    """
    now = datetime.now().isoformat()
    sources = conn.execute("SELECT * FROM sources WHERE active = 1").fetchall()

    print(f"\n{'='*60}")
    print(f"Agent Tick at {now}")
    print(f"{'='*60}")

    for source in sources:
        source = dict(source)
        source_id = source['id']
        source_name = source['name']

        # Check backoff
        failure_count = get_kv(source_id, 'failure_count', 0)
        if failure_count >= 3:
            print(f"\n[{source_name}] Skipped - in backoff (failures: {failure_count})")
            continue

        print(f"\n[{source_name}]")

        try:
            # Fetch content
            content = fetch_content(source_id)
            content_hash = compute_hash(content)

            # Get last snapshot
            last = get_last_snapshot(source_id)

            if last and last['content_hash'] == content_hash:
                print(f"  No change (hash: {content_hash[:12]}...)")
                continue

            # Store new snapshot
            conn.execute(
                "INSERT INTO snapshots (source_id, fetched_at, content_hash, content_text) VALUES (?, ?, ?, ?)",
                (source_id, now, content_hash, content)
            )

            if last is None:
                print(f"  Initial snapshot stored (hash: {content_hash[:12]}...)")
                set_kv(source_id, 'last_alerted_hash', content_hash)
                conn.commit()
                continue

            # Change detected!
            diff_text = compute_diff(last['content_text'], content)

            # Check idempotency: don't alert if already alerted for this hash
            last_alerted = get_kv(source_id, 'last_alerted_hash')
            if last_alerted == content_hash:
                print(f"  Change detected but already alerted (idempotent skip)")
                continue

            # Classify severity
            if use_llm_classification:
                classification = classify_severity_llm(diff_text, source_name)
                severity = classification.severity
                summary = classification.reason
            else:
                severity = classify_severity_heuristic(diff_text)
                summary = f"Content changed ({severity} severity)"

            # Store event
            conn.execute(
                "INSERT INTO events (source_id, created_at, severity, summary, diff_text) VALUES (?, ?, ?, ?, ?)",
                (source_id, now, severity, summary, diff_text)
            )
            set_kv(source_id, 'last_alerted_hash', content_hash)
            set_kv(source_id, 'failure_count', 0)
            conn.commit()

            # Alert
            print(f"  CHANGE DETECTED! Severity: {severity}")
            print(f"  Summary: {summary}")
            print(f"  Diff preview: {diff_text[:200]}...")

        except Exception as e:
            failure_count += 1
            set_kv(source_id, 'failure_count', failure_count)
            conn.commit()
            print(f"  ERROR: {str(e)} (failure #{failure_count})")

    print(f"\n{'='*60}\n")

### Tick 1: Initial Snapshot

The first tick establishes the baseline. Since there are no previous snapshots, the agent stores the initial content for each source without generating any alerts. Think of this as the "seed" step. You should see each source store its first snapshot and nothing more.

In [ ]:
agent_tick()  # Should store initial snapshots, no alerts

### Tick 2: No Changes

Running the agent again immediately should show "No change" for all sources, since the simulated content has not been updated. The hash comparison short-circuits any further processing. Notice how efficient this is: a single SHA-256 comparison is all it takes to confirm nothing has changed.

In [ ]:
agent_tick()  # Should show "no change" for all sources

### Simulate Content Changes

Now let's simulate what happens when monitored pages actually change. We update the OpenAI changelog (adding a breaking deprecation) and the Anthropic models page (adding a new model). The test source remains unchanged.

In [ ]:
# Simulate page updates
current_versions[1] = "v2"  # OpenAI changelog updated
current_versions[2] = "v2"  # Anthropic models page updated
print("Simulated content changes:")
print("  - OpenAI API Changelog: v1 -> v2 (breaking deprecation + new pricing)")
print("  - Anthropic Release Notes: v1 -> v2 (new model added)")
print("  - Local Test Source: unchanged (still v1)")

### Tick 3: Detect Changes with LLM Classification

This tick should detect the changes in sources 1 and 2, compute diffs, classify severity using GPT-5-mini, and store events. Source 3 should show no change. Watch the output carefully to see how the LLM reasons about severity.

In [ ]:
agent_tick(use_llm_classification=True)  # Should detect and classify changes

### Tick 4: Idempotency Check

Running the agent again without any new changes should produce NO new alerts. Even though the content differs from the original v1, the agent has already alerted on the current hash. This demonstrates idempotency, the core property that makes the agent safe to run on unreliable schedules.

In [ ]:
agent_tick()  # Should NOT re-alert (same hashes, already alerted)

### Query All Detected Events

Let's inspect the events table to see all changes the agent has detected and classified. You should see exactly two events from Tick 3.

In [ ]:
print("All detected events:")
print("-" * 60)
events = conn.execute("""
    SELECT e.*, s.name as source_name
    FROM events e JOIN sources s ON e.source_id = s.id
    ORDER BY e.created_at DESC
""").fetchall()

if not events:
    print("No events recorded yet.")
else:
    for event in events:
        event = dict(event)
        print(f"[{event['severity']}] {event['source_name']}: {event['summary']}")
        print(f"  Created: {event['created_at']}")
        print()

## Approach 2: Push-Based Mechanism

The stateful worker (Approach 1) uses a **pull-based** model: it polls sources on a schedule whether or not anything has changed. This is simple and works for any external page, but it wastes resources when nothing changes and has latency equal to the polling interval.

The **push-based** approach flips this: instead of polling, the agent exposes an endpoint (webhook) that receives notifications when content changes. This gives:

- **Near real-time** detection (no polling delay)
- **Zero wasted work** (only processes actual changes)
- **Lower resource usage** (no unnecessary fetches)

The trade-off is that the monitored system must support webhooks or change notifications. Many modern platforms (GitHub, Stripe, Slack) do, but arbitrary web pages do not.

Below we implement a `WebhookHandler` class that simulates this pattern. In production, this would be an HTTP server (e.g., Flask or FastAPI) that receives POST requests.

In [ ]:
class WebhookHandler:
    """
    Simulates a push-based webhook handler.
    In production, this would be an HTTP endpoint that receives
    POST requests when a monitored page changes.
    """
    def __init__(self, db_conn, openai_client):
        self.conn = db_conn
        self.client = openai_client

    def handle_webhook(self, payload):
        """
        Process an incoming webhook notification.

        Args:
            payload: dict with keys:
                - source_id (int): Which source changed
                - new_content (str): The updated page content
                - timestamp (str, optional): When the change occurred

        Returns:
            dict with status and details about what happened.
        """
        source_id = payload['source_id']
        new_content = payload['new_content']
        timestamp = payload.get('timestamp', datetime.now().isoformat())

        source = dict(self.conn.execute(
            "SELECT * FROM sources WHERE id = ?", (source_id,)
        ).fetchone())

        new_hash = compute_hash(new_content)
        last = get_last_snapshot(source_id)

        # Idempotency check
        if last and last['content_hash'] == new_hash:
            return {"status": "no_change", "message": "Content unchanged"}

        # Store snapshot
        self.conn.execute(
            "INSERT INTO snapshots (source_id, fetched_at, content_hash, content_text) VALUES (?, ?, ?, ?)",
            (source_id, timestamp, new_hash, new_content)
        )

        if last is None:
            self.conn.commit()
            return {"status": "initial", "message": "Initial snapshot stored"}

        # Process change
        diff_text = compute_diff(last['content_text'], new_content)
        classification = classify_severity_llm(diff_text, source['name'])

        self.conn.execute(
            "INSERT INTO events (source_id, created_at, severity, summary, diff_text) VALUES (?, ?, ?, ?, ?)",
            (source_id, timestamp, classification.severity, classification.reason, diff_text)
        )
        self.conn.commit()

        return {
            "status": "change_detected",
            "severity": classification.severity,
            "summary": classification.reason
        }


handler = WebhookHandler(conn, client)
print("WebhookHandler initialized and ready to receive events.")

### Test the Push-Based Handler

Let's simulate receiving a webhook notification for the test source with new content that includes a breaking change keyword. The handler should detect the change, classify its severity, and store an event.

In [ ]:
# Simulate receiving a webhook for the test source
webhook_content = simulated_content[3]["v2"]
result = handler.handle_webhook({
    "source_id": 3,
    "new_content": webhook_content,
    "timestamp": datetime.now().isoformat()
})
print(f"Webhook result: {json.dumps(result, indent=2)}")

In [ ]:
# Verify the event was stored
print("Updated events list (including push-based event):")
print("-" * 60)
events = conn.execute("""
    SELECT e.*, s.name as source_name
    FROM events e JOIN sources s ON e.source_id = s.id
    ORDER BY e.created_at DESC
""").fetchall()

for event in events:
    event = dict(event)
    print(f"[{event['severity']}] {event['source_name']}: {event['summary']}")
    print(f"  Created: {event['created_at']}")
    print()

## Comparing the Two Approaches

| Aspect | Stateful Worker (Pull) | Push-Based (Webhook) |
|--------|----------------------|---------------------|
| **Trigger** | Timer/schedule | External event |
| **Latency** | Depends on poll interval | Near real-time |
| **Complexity** | Simpler to implement | Needs webhook infrastructure |
| **Resource usage** | Polls even when nothing changes | Only processes on change |
| **Best for** | Monitoring external pages | Systems that support webhooks |
| **Idempotency** | Hash-based deduplication | Hash-based deduplication |
| **Error handling** | Backoff on repeated failures | Retry on webhook delivery failure |

In practice, many production systems use a **hybrid approach**: webhooks for systems that support them, and polling for external pages that do not. Both approaches share the same core logic (hash, diff, classify, store); only the trigger mechanism differs.

## Your Turn: Build a Price Monitor Agent

Same stateful loop pattern, but applied to price monitoring instead of documentation watching. You will adapt the tick-based architecture to detect and classify price changes.

## Exercise 2: Build a Price Monitor Agent

In the Docs Watcher notebook you built a stateful agent that polls pages, detects changes via content hashing, classifies severity, and stores events in a database. In this exercise you will adapt that same pattern to **product price monitoring**.

Your agent will:
1. Poll a set of product price sources on each tick.
2. Hash the price data to detect changes efficiently.
3. When a price changes, compute the percentage difference and classify the significance.
4. Store price change events for querying later.

The infrastructure (database schema, helper functions, simulated data) is provided for you. You need to implement the classification logic and the main agent tick function.

In [ ]:
import mermaid as md

md.Mermaid("""
flowchart LR
    A[Tick] --> B[Fetch Prices]
    B --> C[Hash Data]
    C --> D{Compare to Last Hash}
    D -- No Change --> E[Log & Skip]
    D -- Change Detected --> F[Compute % Change]
    F --> G[Classify Significance]
    G --> H[Store Event & Alert]
""")

### Database and Helper Functions (Provided)

In [ ]:
import sqlite3
import hashlib
import json
from datetime import datetime

# Create in-memory database
price_conn = sqlite3.connect(":memory:")
price_conn.row_factory = sqlite3.Row

price_conn.executescript("""
CREATE TABLE price_sources (
    id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    retailer TEXT NOT NULL,
    active BOOLEAN DEFAULT 1
);

CREATE TABLE price_snapshots (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_id INTEGER,
    fetched_at TEXT,
    content_hash TEXT,
    price REAL,
    raw_data TEXT,
    FOREIGN KEY (source_id) REFERENCES price_sources(id)
);

CREATE TABLE price_events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_id INTEGER,
    created_at TEXT,
    severity TEXT,
    old_price REAL,
    new_price REAL,
    pct_change REAL,
    direction TEXT,
    summary TEXT,
    FOREIGN KEY (source_id) REFERENCES price_sources(id)
);

CREATE TABLE kv (
    source_id INTEGER,
    key TEXT,
    value TEXT,
    PRIMARY KEY (source_id, key),
    FOREIGN KEY (source_id) REFERENCES price_sources(id)
);
""")

print("Database schema created: price_sources, price_snapshots, price_events, kv")

In [ ]:
def compute_hash(content):
    """Compute SHA-256 hash of content for change detection."""
    return hashlib.sha256(content.encode()).hexdigest()


def get_last_price_snapshot(source_id):
    """Retrieve the most recent price snapshot for a source."""
    row = price_conn.execute(
        "SELECT * FROM price_snapshots WHERE source_id = ? ORDER BY fetched_at DESC LIMIT 1",
        (source_id,)
    ).fetchone()
    return dict(row) if row else None


def get_kv(source_id, key, default=None):
    """Get a value from the per-source key-value store."""
    row = price_conn.execute(
        "SELECT value FROM kv WHERE source_id = ? AND key = ?",
        (source_id, key)
    ).fetchone()
    return json.loads(row[0]) if row else default


def set_kv(source_id, key, value):
    """Set a value in the per-source key-value store."""
    price_conn.execute(
        "INSERT OR REPLACE INTO kv (source_id, key, value) VALUES (?, ?, ?)",
        (source_id, key, json.dumps(value))
    )
    price_conn.commit()


print("Helper functions defined: compute_hash, get_last_price_snapshot, get_kv, set_kv")

### Simulated Price Data

Just like the Docs Watcher used simulated page content, we use simulated price data with two versions per product. All products start at `v1`. Later you will flip them to `v2` to trigger price change detection.

In [ ]:
simulated_prices = {
    1: {
        "v1": {"product": "GPU RTX 5090", "price": 1999.99, "retailer": "TechStore"},
        "v2": {"product": "GPU RTX 5090", "price": 1799.99, "retailer": "TechStore"},
    },
    2: {
        "v1": {"product": "AI Dev Kit Pro", "price": 499.99, "retailer": "DevShop"},
        "v2": {"product": "AI Dev Kit Pro", "price": 599.99, "retailer": "DevShop"},
    },
    3: {
        "v1": {"product": "Cloud Server (monthly)", "price": 89.99, "retailer": "CloudHost"},
        "v2": {"product": "Cloud Server (monthly)", "price": 87.99, "retailer": "CloudHost"},
    },
}

# Track which version each source is on
price_versions = {sid: "v1" for sid in simulated_prices}

# Register sources in the database
for sid, versions in simulated_prices.items():
    v1 = versions["v1"]
    price_conn.execute(
        "INSERT INTO price_sources (id, product_name, retailer) VALUES (?, ?, ?)",
        (sid, v1["product"], v1["retailer"])
    )
price_conn.commit()


def fetch_price(source_id):
    """Simulate fetching the current price data for a source."""
    version = price_versions.get(source_id, "v1")
    return simulated_prices.get(source_id, {}).get(version, {})


print("Simulated prices loaded. All sources start at v1.")
for sid, versions in simulated_prices.items():
    v1 = versions["v1"]
    v2 = versions["v2"]
    print(f"  [{sid}] {v1['product']} @ {v1['retailer']}: "
          f"v1=${v1['price']:.2f}, v2=${v2['price']:.2f}")

### Step 1: `classify_price_change(old_price, new_price)`

Implement the classification logic for price changes:

| Condition | Severity |
|-----------|----------|
| Price drop greater than 10% | **HIGH** |
| Price drop between 5% and 10% | **MEDIUM** |
| Price drop less than 5% | **LOW** |
| Any price increase | **MEDIUM** |

Return a dictionary with `severity`, `direction` ("drop" or "increase"), and `pct_change` (as a positive percentage).

In [ ]:
def classify_price_change(old_price, new_price):
    """
    Classify the significance of a price change.

    Returns a dict with:
        - severity: "HIGH", "MEDIUM", or "LOW"
        - direction: "drop" or "increase"
        - pct_change: absolute percentage change (positive number)
    """
    # YOUR CODE HERE
    # Hint:
    # 1. Calculate the percentage change: abs(new_price - old_price) / old_price * 100
    # 2. Determine direction: "drop" if new_price < old_price, else "increase"
    # 3. Apply the severity rules from the table above.
    # 4. Return {"severity": ..., "direction": ..., "pct_change": ...}
    pass

### Step 2: `price_monitor_tick()` (The Agent Tick Function)

This is the main loop, directly analogous to `agent_tick()` from the Docs Watcher. For each active source:

1. Fetch the current price data.
2. Serialize it to JSON and compute its hash.
3. Compare the hash to the last stored snapshot.
4. If unchanged, log and skip.
5. If this is the first snapshot, store it as a baseline.
6. If changed, compute the price difference, classify it, store a `price_events` row, and print an alert.
7. Use the `kv` table to track `last_alerted_hash` for idempotency.

In [ ]:
def price_monitor_tick():
    """
    One tick of the price monitor agent.
    Fetches all active sources, checks for price changes, and creates events.
    """
    now = datetime.now().isoformat()
    sources = price_conn.execute(
        "SELECT * FROM price_sources WHERE active = 1"
    ).fetchall()

    print(f"\n{'='*60}")
    print(f"Price Monitor Tick at {now}")
    print(f"{'='*60}")

    for source in sources:
        source = dict(source)
        source_id = source["id"]
        product_name = source["product_name"]

        print(f"\n[{product_name}]")

        # YOUR CODE HERE
        # Follow the same structure as agent_tick() from the Docs Watcher:
        #
        # 1. price_data = fetch_price(source_id)
        # 2. raw = json.dumps(price_data, sort_keys=True)
        # 3. content_hash = compute_hash(raw)
        # 4. last = get_last_price_snapshot(source_id)
        # 5. If last exists and hashes match -> print "  No change" and continue
        # 6. Store new snapshot in price_snapshots table
        # 7. If no previous snapshot -> print "  Initial snapshot" and continue
        # 8. Check idempotency: if get_kv(source_id, "last_alerted_hash") == content_hash -> skip
        # 9. old_price = last["price"], new_price = price_data["price"]
        # 10. classification = classify_price_change(old_price, new_price)
        # 11. Build a summary string, e.g. "Price drop of 10.0%: $1999.99 -> $1799.99"
        # 12. Insert into price_events table
        # 13. set_kv(source_id, "last_alerted_hash", content_hash)
        # 14. Print the alert with severity, direction, and prices
        pass

    print(f"\n{'='*60}\n")

### Test Your Price Monitor

Run the cells below in order. Tick 1 stores initial snapshots. Then we simulate price changes and run Tick 2 to verify that your agent detects them correctly.

**Tick 1: Establish baseline snapshots**

In [ ]:
price_monitor_tick()  # Should store initial snapshots for all 3 products

**Verify: Tick 1 should show no events yet**

In [ ]:
events = price_conn.execute("SELECT COUNT(*) as cnt FROM price_events").fetchone()
print(f"Events after Tick 1: {events['cnt']} (expected: 0)")

**Simulate price changes, then run Tick 2**

In [ ]:
# Flip sources to v2
price_versions[1] = "v2"  # GPU: $1999.99 -> $1799.99 (10% drop, HIGH)
price_versions[2] = "v2"  # Dev Kit: $499.99 -> $599.99 (20% increase, MEDIUM)
# Source 3 stays at v1 (Cloud Server unchanged)

print("Simulated price changes:")
print("  GPU RTX 5090: $1999.99 -> $1799.99 (price drop)")
print("  AI Dev Kit Pro: $499.99 -> $599.99 (price increase)")
print("  Cloud Server: unchanged")

In [ ]:
price_monitor_tick()  # Should detect changes in sources 1 and 2, skip source 3

**Query all detected price events**

In [ ]:
print("All detected price events:")
print("-" * 60)
events = price_conn.execute("""
    SELECT e.*, s.product_name, s.retailer
    FROM price_events e
    JOIN price_sources s ON e.source_id = s.id
    ORDER BY e.created_at DESC
""").fetchall()

if not events:
    print("No events recorded. Check your price_monitor_tick() implementation.")
else:
    for event in events:
        event = dict(event)
        print(f"[{event['severity']}] {event['product_name']} @ {event['retailer']}")
        print(f"  {event['summary']}")
        print(f"  ${event['old_price']:.2f} -> ${event['new_price']:.2f} "
              f"({event['pct_change']:.1f}% {event['direction']})")
        print()

**Idempotency check: Tick 3 should produce no new events**

In [ ]:
price_monitor_tick()  # Should show "No change" for all sources

events_after = price_conn.execute("SELECT COUNT(*) as cnt FROM price_events").fetchone()
print(f"\nTotal events after Tick 3: {events_after['cnt']} (should still be 2)")

## Summary

In this notebook, you built a **Docs Watcher Agent** that demonstrates long-running agent loops with state persistence. Here are the key takeaways:

- **Stateful agents need persistent storage** (SQLite, Redis, PostgreSQL, etc.) to remember what they have seen across execution cycles.
- **Idempotency prevents duplicate alerts**: By tracking the last alerted hash per source, re-running the agent is always safe. Content hashing with SHA-256 makes change detection efficient.
- **Severity classification helps prioritize attention**: Combining fast heuristic rules with optional LLM classification gives both speed and accuracy.
- **The stateful worker pattern (poll, diff, classify, alert) is a universal monitoring primitive** that applies far beyond documentation watching.
- **Push-based mechanisms are more efficient when webhooks are available**, eliminating wasted polls and reducing detection latency to near real-time.
- **Backoff logic prevents hammering failed sources**, making the agent robust against transient and persistent failures.
- **This pattern generalizes to many domains**: security monitoring, competitor tracking, compliance checking, API changelog monitoring, price tracking, and more.